# Python _in_ Electrical Engineering
## Electrical Circuits Simulations with PySpice
1. Install PySpice and Ngspice and/or verify your setup.
2. Simulate the step response of an RC circuit (R = 1 kΩ, C = 1 μF, input 0→5 V) and an RL circuit (R = 10 Ω, L = 10 mH, input 0→10 V). Plot input voltage voltages at L, C components _in_ these circuits. 
3. Simulate a series RLC circuit (R = 50 Ω, L = 100 mH, C = 10 μF) with a sinusoidal source (e.g., 50 Hz), measure voltage and current, and compute active power (P), reactive power (Q), and apparent power (S), commenting on phase shift and whether the circuit is inductive or capacitive.
4. Extend the RLC circuit analysis by introducing power factor correction: add a compensating capacitor (or inductor) to reduce reactive power, re-simulate the circuit, and compare P, Q, S, and power factor before and after compensation, explaining how compensation improves energy efficiency.
5. Prepare a report including circuit schematics, simulation plots, calculated values, and a short discussion of your results and conclusions.

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import PySpice.Logging.Logging as Logging
from PySpice.Spice.Netlist import Circuit
from PySpice.Unit import *
from PySpice.Spice.NgSpice.Shared import NgSpiceShared
logger = Logging.setup_logging(logging_level=logging.DEBUG)

#print(logger)
# check the instalation of pyspice

In [ ]:
original_send_char = NgSpiceShared._send_char

def patched_send_char(self, message, message_id):
    msg_str = message.decode('utf-8', errors='ignore') if isinstance(message, bytes) else str(message)
    
    if "Using SPARSE" in msg_str or "compatibility mode" in msg_str:
        return 0 
        
    return original_send_char(self, message, message_id)

# REPLACE THE STDERR DATA
NgSpiceShared._send_char = patched_send_char

In [17]:
# --- RC circuit ---
circuit_rc = Circuit('RC Step Response')
# (0 to 5V, start 0s, 10us, time 5ms)
circuit_rc.PulseVoltageSource('input', '_in_', circuit_rc.gnd,
                              initial_value=0@u_V, pulsed_value=5@u_V,
                              delay_time=0@u_s, rise_time=1@u_us, fall_time=1@u_us,
                              pulse_width=5@u_ms, period=10@u_ms)
circuit_rc.R(1, '_in_', 'out', 1@u_kOhm)
circuit_rc.C(1, 'out', circuit_rc.gnd, 1@u_uF)

simulator_rc = circuit_rc.simulator(temperature=25, nominal_temperature=25, simulator='ngspice-shared')
analysis_rc = simulator_rc.transient(step_time=10@u_us, end_time=5@u_ms)

# --- RL circuit ---
circuit_rl = Circuit('RL Step Response')
circuit_rl.PulseVoltageSource('input', '_in_', circuit_rl.gnd,
                              initial_value=0@u_V, pulsed_value=10@u_V,
                              delay_time=0@u_s, rise_time=1@u_us, fall_time=1@u_us,
                              pulse_width=5@u_ms, period=10@u_ms)
circuit_rl.R(1, '_in_', 'out', 10@u_Ohm)
circuit_rl.L(1, 'out', circuit_rl.gnd, 10@u_mH)

simulator_rl = circuit_rl.simulator(temperature=25, nominal_temperature=25, simulator='ngspice-shared')
analysis_rl = simulator_rl.transient(step_time=10@u_us, end_time=5@u_ms)

# --- plots ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# RC circuit plot
time_rc = np.array(analysis_rc.time) * 1000 # miliseconds
ax1.plot(time_rc, np.array(analysis_rc['_in_']), label='V_input (5V)')
ax1.plot(time_rc, np.array(analysis_rc['out']), label='V_C (capacitor voltage)')
ax1.set_title('RC step response')
ax1.set_xlabel('time [ms]')
ax1.set_ylabel('Voltage [V]')
ax1.grid()
ax1.legend()

# RL circuit plot 
time_rl = np.array(analysis_rl.time) * 1000
ax2.plot(time_rl, np.array(analysis_rl['_in_']), label='V_input (10V)')
ax2.plot(time_rl, np.array(analysis_rl['_in_']) - np.array(analysis_rl['out']), label='V_L (coil voltage)')
ax2.set_title('RL step response')
ax2.set_xlabel('Time [ms]')
ax2.set_ylabel('Voltage [V]')
ax2.grid()
ax2.legend()

plt.tight_layout()
plt.show()

NgSpiceCommandError: Command 'run' failed